# CNN + BiLSTM Single-User Training (Colab)
Two 1-D conv blocks (stride 2) + 2-layer BiLSTM, CTC loss. Uses subject #89335547.


## 0. Runtime check

In [ ]:
import torch, platform
print('python', platform.python_version())
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))


## 1. Install deps (repo root)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .


## 2. Data (subject #89335547)
If already at /content/data, skip; otherwise download+rename and symlink.


In [ ]:
import os, pathlib, subprocess
DATA_ROOT = pathlib.Path('/content/emg2qwerty-data-2021-08')
if not DATA_ROOT.exists():
    os.makedirs('/content/tmp_dl', exist_ok=True)
    tar_path = '/content/tmp_dl/emg2qwerty-data-2021-08.tar.gz'
    url = 'https://fb-ctrl-oss.s3.amazonaws.com/emg2qwerty/emg2qwerty-data-2021-08.tar.gz'
    subprocess.run(['bash','-lc', f"curl -L {url} -o {tar_path}"], check=True)
    subprocess.run(['bash','-lc', f"tar -xzf {tar_path} -C /content --wildcards 'emg2qwerty-data-2021-08/metadata.csv' 'emg2qwerty-data-2021-08/89335547_*.hdf5'"], check=True)
    pathlib.Path('/content/emg2qwerty-data-2021-08').rename(DATA_ROOT)
for f in DATA_ROOT.glob('89335547_*.hdf5'):
    tgt = f.with_name(f.name.replace('89335547_', '', 1))
    if not tgt.exists():
        f.rename(tgt)
link = pathlib.Path('data')
if link.exists() or link.is_symlink():
    link.unlink()
link.symlink_to(DATA_ROOT)
!ls -lh data | head


## 3. Train CNN+BiLSTM model

In [ ]:
!python -m emg2qwerty.train   model=cnn_lstm_ctc   user=single_user   dataset.root=/content/data   trainer.accelerator=gpu trainer.devices=1   trainer.max_epochs=40   batch_size=32   num_workers=2


## 4. 1-epoch sanity check

In [ ]:
!python -m emg2qwerty.train   model=cnn_lstm_ctc   user=single_user   dataset.root=/content/data   trainer.accelerator=gpu trainer.devices=1   trainer.max_epochs=1   batch_size=8   num_workers=2
